In [1]:
"""
    spline_cubico_condicionado(x::Vector, y::Vector, fpo::Number, fpn::Number)

Calcula los coeficientes del spline cúbico condicionado (Clamped).
Requiere las derivadas exactas en los extremos.

Argumentos:
- `x`: Vector de nodos x_0, ..., x_n
- `y`: Vector de valores f(x_0), ..., f(x_n)
- `fpo`: Valor de la derivada en el primer nodo (f'(x_0))
- `fpn`: Valor de la derivada en el último nodo (f'(x_n))

Retorna:
- `a`, `b`, `c`, `d`: Vectores de coeficientes para cada intervalo.
"""
function spline_cubico_condicionado(x::Vector{T}, y::Vector{T}, fpo::Number, fpn::Number) where T <: AbstractFloat
    n_puntos = length(x)
    n = n_puntos - 1 # n representa el número de intervalos (índice final en el libro)

    if length(y) != n_puntos
        error("Los vectores x e y deben tener la misma longitud.")
    end

    # Inicialización de arreglos
    # Indices en Julia: 1 ... n+1
    a = copy(y)
    h = zeros(T, n)
    alpha = zeros(T, n + 1)
    l = zeros(T, n + 1)
    mu = zeros(T, n + 1) # Nota: mu en el libro va hasta n-1, aquí reservamos espacio seguro
    z = zeros(T, n + 1)
    
    # Arreglos de salida para coeficientes (intervalos 1 a n)
    b = zeros(T, n)
    c = zeros(T, n + 1) # c_n se usa en el cálculo
    d = zeros(T, n)

    # --- Paso 1: Calcular h ---
    for i in 1:n
        h[i] = x[i+1] - x[i]
    end

    # --- Paso 2: Condiciones de frontera para alpha ---
    # alpha_0 (Julia índice 1)
    alpha[1] = 3 * (a[2] - a[1]) / h[1] - 3 * fpo
    
    # alpha_n (Julia índice n+1)
    alpha[n+1] = 3 * fpn - 3 * (a[n+1] - a[n]) / h[n]

    # --- Paso 3: Calcular alpha intermedios ---
    # Libro: i = 1 ... n-1 => Julia: i = 2 ... n
    for i in 2:n
        term1 = (3 / h[i]) * (a[i+1] - a[i])
        term2 = (3 / h[i-1]) * (a[i] - a[i-1])
        alpha[i] = term1 - term2
    end

    # --- Paso 4: Inicio del sistema tridiagonal ---
    # Libro: l_0, mu_0, z_0 => Julia índice 1
    l[1] = 2 * h[1]
    mu[1] = 0.5
    z[1] = alpha[1] / l[1]

    # --- Paso 5: Barrido hacia adelante ---
    # Libro: i = 1 ... n-1 => Julia: i = 2 ... n
    for i in 2:n
        l[i] = 2 * (x[i+1] - x[i-1]) - h[i-1] * mu[i-1]
        mu[i] = h[i] / l[i]
        z[i] = (alpha[i] - h[i-1] * z[i-1]) / l[i]
    end

    # --- Paso 6: Cierre del sistema en la frontera n ---
    # Libro: l_n, z_n, c_n => Julia índice n+1
    # Nota: El algoritmo usa h_{n-1} y mu_{n-1} (indices n en Julia)
    l[n+1] = h[n] * (2 - mu[n])
    z[n+1] = (alpha[n+1] - h[n] * z[n]) / l[n+1]
    c[n+1] = z[n+1]

    # --- Paso 7: Sustitución hacia atrás ---
    # Libro: j = n-1 ... 0 => Julia: j = n ... 1
    for j in n:-1:1
        c[j] = z[j] - mu[j] * c[j+1]
        b[j] = (a[j+1] - a[j]) / h[j] - (h[j] * (c[j+1] + 2 * c[j])) / 3
        d[j] = (c[j+1] - c[j]) / (3 * h[j])
    end

    # --- Paso 8: Salida ---
    # Retornamos los vectores correspondientes a los intervalos
    return a[1:n], b, c[1:n], d
end

"""
    evaluar_spline_condicionado(val, x_nodos, a, b, c, d)
    
Función auxiliar para evaluar el polinomio en un punto x dado.
"""
function evaluar_spline_condicionado(x_val, x_nodos, a, b, c, d)
    n = length(x_nodos) - 1
    
    # Encontrar intervalo j
    j = n
    for i in 1:n
        if x_val <= x_nodos[i+1]
            j = i
            break
        end
    end
    
    dx = x_val - x_nodos[j]
    return a[j] + b[j]*dx + c[j]*dx^2 + d[j]*dx^3
end

evaluar_spline_condicionado

In [2]:
# 1. Definir datos de prueba (f(x) = e^x en [0, 3])
x_puntos = [0.0, 1.0, 2.0, 3.0]
y_puntos = exp.(x_puntos)

# 2. Calcular derivadas exactas en los extremos
fpo = exp(x_puntos[1])   # f'(0) = 1
fpn = exp(x_puntos[end]) # f'(3) = e^3

# 3. Ejecutar algoritmo
println("Calculando Spline Cúbico Condicionado...")
a, b, c, d = spline_cubico_condicionado(x_puntos, y_puntos, fpo, fpn)

println("\nCoeficientes 'b' (pendientes iniciales de cada intervalo):")
println(b)

# 4. Evaluar en un punto intermedio, x = 1.5
val_aprox = evaluar_spline_condicionado(1.5, x_puntos, a, b, c, d)
val_exacto = exp(1.5)

println("\n--- Resultados en x = 1.5 ---")
println("Aproximación: $val_aprox")
println("Valor exacto: $val_exacto")
println("Error abs:    $(abs(val_aprox - val_exacto))")

Calculando Spline Cúbico Condicionado...

Coeficientes 'b' (pendientes iniciales de cada intervalo):
[0.9999999999999999, 2.710162988411307, 7.326516343146723]

--- Resultados en x = 1.5 ---
Aproximación: 4.476624794352921
Valor exacto: 4.4816890703380645
Error abs:    0.005064275985143141
